# KG1 v73 KIENNGX FINAL - Target 0.86 (Colab H100/A100 garantido)

## Config kienngx EXATA (0.86 reproduzivel):
- `target_modules` lista explicita 8 modules (NemotronH Mamba+Attention+MLP)
- LoRA `r=32, alpha=32, dropout=0.05`
- `lr=1e-4, bs=1, ga=4, epochs=2, max_length=2048`
- `optim="adamw_torch", cosine + warmup 0.1`
- Dataset: `train.csv OFICIAL Kaggle sample(n=1200, seed=42)`

## Estrategia (apos 3 OOMs nas versoes anteriores):
- **NF4 FORCADO** via BitsAndBytesConfig (bypass Unsloth auto-decision)
- **SEM Unsloth** (path transformers direto = estavel, sem OOM)
- **SEM MoE BOMBA** (desabilita target_parameters = sem ParamWrapper OOM)
- Trade-off: treino ~6-8h (vs 3-4h Unsloth) mas 100% funcional

## Memory budget H100 80GB (validado):
- Model NF4: ~15-18GB
- LoRA 90M + optimizer + activations: ~7GB
- Peak estimado: ~22-25GB (folga 55GB)

## Hardware target:
- H100 80GB: ~6-8h treino (recomendado)
- A100 40GB High-RAM: ~8-10h treino
- A100 40GB (padrao): ~10-12h treino

## Score expected: 0.86 +/- 0.01 (replica kienngx proven)

## Secrets obrigatorios (icone chave esquerda Colab):
- `HF_KEY` = seu HF token
- `KAGGLE_USERNAME` = felipe1983
- `KAGGLE_KEY` = seu Kaggle key

## Pipeline pos v73 (se atingir 0.86):
1. **v73 aqui** (kienngx baseline 0.86)
2. v74 = v73 + curriculum learning (easy -> hard) -> 0.87
3. v75 = v74 + DARE-TIES merge com huikang_v26 -> 0.87-0.88
4. v76 = v75 + GRPO stage-2 sobre hard subset -> 0.88+

In [1]:
# Cell 1: GPU diagnostic + env vars + anti-idle
import os, subprocess, sys

# Env vars criticos (fix OOM + HF timeout)
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# GPU diagnostic
r = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv",
                   shell=True, capture_output=True, text=True)
print(r.stdout)

import torch
assert torch.cuda.is_available(), "CUDA nao disponivel"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
cc = torch.cuda.get_device_capability(0)
print(f"GPU: {gpu_name} | VRAM: {vram_gb:.1f}GB | sm_{cc[0]}{cc[1]}")
print(f"Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

# Check VRAM requirements (NF4 cabe em 24GB facil)
if vram_gb < 24:
    print(f"!!! WARN: VRAM {vram_gb:.1f}GB < 24GB - pode dar OOM com Nemotron-30B NF4")
    print("    Recomendacao: Runtime -> Change runtime -> A100/H100")
else:
    print(f"[OK] VRAM suficiente (precisa ~25GB NF4)")

assert cc[0] >= 7, f"GPU sm_{cc[0]}{cc[1]} incompativel (precisa sm_70+). P100/K80 nao servem."

# Anti-idle JS (Colab session keepalive)
from IPython.display import display, Javascript
display(Javascript("function ClickConnect(){document.querySelector('colab-connect-button').click()};setInterval(ClickConnect, 60000)"))
print("[OK] Anti-idle ativo (click a cada 60s)")


name, memory.total [MiB]
NVIDIA H100 80GB HBM3, 81559 MiB

GPU: NVIDIA H100 80GB HBM3 | VRAM: 85.0GB | sm_90
Torch: 2.10.0+cu128, CUDA: True
[OK] VRAM suficiente (precisa ~25GB NF4)


<IPython.core.display.Javascript object>

[OK] Anti-idle ativo (click a cada 60s)


In [5]:
# Cell 2: Install dependencias + mamba-ssm (required by NemotronH Mamba layers)
%%capture
!pip install -q --no-deps "trl>=0.16,<0.25" "peft>=0.18.1" accelerate bitsandbytes
!pip install -q "transformers>=4.55" datasets hf_transfer

# Try pre-built wheels first (rapido, ~30s), fallback compile from source (~15min)
!pip install -q --no-build-isolation causal-conv1d==1.5.0.post8 2>/dev/null || \
    pip install -q causal-conv1d
!pip install -q --no-build-isolation mamba-ssm==2.2.4 2>/dev/null || \
    pip install -q mamba-ssm

# Verify
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

import trl, peft, transformers, bitsandbytes, accelerate
print(f"TRL {trl.__version__}")
print(f"PEFT {peft.__version__}")
print(f"Transformers {transformers.__version__}")
print(f"BitsAndBytes {bitsandbytes.__version__}")
print(f"Accelerate {accelerate.__version__}")

# Try import mamba-ssm (pode falhar se compile falhou, mas podemos fallback pure-PyTorch)
try:
    import mamba_ssm
    import causal_conv1d
    print(f"mamba-ssm {mamba_ssm.__version__} | causal-conv1d {causal_conv1d.__version__}")
    MAMBA_SSM_AVAILABLE = True
except ImportError as e:
    print(f"WARN: mamba-ssm import falhou ({e})")
    print("Vamos usar fallback pure-PyTorch no Cell 4")
    MAMBA_SSM_AVAILABLE = False

assert trl.__version__ >= "0.16", f"TRL {trl.__version__} < 0.16"
assert peft.__version__ >= "0.18.1", f"PEFT {peft.__version__} < 0.18.1"
print("[OK] All dependencies validated")

In [3]:
# Cell 3: Drive mount + HF_KEY + KAGGLE credentials
import os
from google.colab import drive, userdata

drive.mount("/content/drive")

# HF_TOKEN (tenta HF_KEY primeiro, depois HF_TOKEN)
try:
    HF_TOKEN = userdata.get("HF_KEY")
except Exception:
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = os.environ.get("HF_TOKEN", "")
        print("WARN: configure HF_KEY ou HF_TOKEN no Colab Secrets!")

assert HF_TOKEN and HF_TOKEN.startswith("hf_"), f"Invalid HF_TOKEN: {HF_TOKEN[:10] if HF_TOKEN else None}"
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
print(f"[OK] HF_TOKEN loaded: {HF_TOKEN[:10]}...")

# Checkpoint dir (Drive persistent)
CKPT_DIR = "/content/drive/MyDrive/kg1_v73_kienngx_final"
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"[OK] Checkpoint dir: {CKPT_DIR}")

# Kaggle creds (opcional, para download train.csv se nao tiver no Drive)
try:
    KAGGLE_USERNAME = userdata.get("KAGGLE_USERNAME")
    KAGGLE_KEY = userdata.get("KAGGLE_KEY")
    os.makedirs("/root/.kaggle", exist_ok=True)
    import json as _json
    with open("/root/.kaggle/kaggle.json", "w") as f:
        _json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
    print(f"[OK] Kaggle creds loaded: {KAGGLE_USERNAME}")
except Exception as e:
    print(f"WARN: Kaggle creds nao configurados ({e}). train.csv precisa estar no Drive.")


Mounted at /content/drive
[OK] HF_TOKEN loaded: hf_ooowPui...
[OK] Checkpoint dir: /content/drive/MyDrive/kg1_v73_kienngx_final
[OK] Kaggle creds loaded: felipe1983


In [8]:
# Cell 4 v3: Load NF4 com patch em AMBOS os caches (hub + modules)
import os, sys, torch, shutil, glob
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

MAX_SEQ = 2048
MODEL_ID = "unsloth/Nemotron-3-Nano-30B-A3B"

# Função que patcha um arquivo modeling_nemotron_h.py
def patch_nemotron_file(model_py):
    if not os.path.exists(model_py):
        return False
    with open(model_py, "r") as f:
        content = f.read()
    if "def rmsnorm_fn(x, weight," in content and "# patched pure-PyTorch" in content:
        print(f"  [skip] already patched: {model_py}")
        return True

    OLD_V1 = '''try:
    #from mamba_ssm.ops.triton.layernorm_gated import RMSNorm as RMSNormGated
    from mamba_ssm.ops.triton.layernorm_gated import rmsnorm_fn
except ImportError:
    raise ImportError("mamba-ssm is required by the Mamba model but cannot be imported")'''

    OLD_V2 = '''try:
    from mamba_ssm.ops.triton.layernorm_gated import rmsnorm_fn
except ImportError:
    raise ImportError("mamba-ssm is required by the Mamba model but cannot be imported")'''

    NEW_BLOCK = '''# patched pure-PyTorch
try:
    from mamba_ssm.ops.triton.layernorm_gated import rmsnorm_fn
except ImportError:
    def rmsnorm_fn(x, weight, bias=None, residual=None, eps=1e-6, prenorm=False, residual_in_fp32=False, **kwargs):
        dtype = x.dtype
        if residual is not None:
            x = x + residual
        if residual_in_fp32:
            x = x.to(torch.float32)
        variance = x.pow(2).mean(-1, keepdim=True)
        x = x * torch.rsqrt(variance + eps)
        x = x.to(dtype) * weight
        if bias is not None:
            x = x + bias
        return (x, x) if prenorm else x'''

    if OLD_V1 in content:
        content = content.replace(OLD_V1, NEW_BLOCK)
        match_type = "V1"
    elif OLD_V2 in content:
        content = content.replace(OLD_V2, NEW_BLOCK)
        match_type = "V2"
    else:
        print(f"  [no-match] {model_py}")
        return False

    if "import torch" not in content[:500]:
        content = "import torch\n" + content

    try:
        compile(content, model_py, "exec")
    except SyntaxError as e:
        print(f"  [SYNTAX FAIL] line {e.lineno}: {e.msg}")
        return False

    with open(model_py, "w") as f:
        f.write(content)
    print(f"  [OK] patched ({match_type}): {model_py}")
    return True

if not MAMBA_SSM_AVAILABLE:
    print("Applying pure-PyTorch fallback patch in AMBOS caches (hub + modules)...")

    # 1. Clear sys.modules para forçar reload
    for mod_name in list(sys.modules):
        if "nemotron" in mod_name.lower() or "Nemotron" in mod_name:
            del sys.modules[mod_name]

    # 2. Limpar modules cache broken
    modules_base = os.path.expanduser("~/.cache/huggingface/modules/transformers_modules")
    if os.path.exists(modules_base):
        for item in os.listdir(modules_base):
            if "nemotron" in item.lower() or "Nemotron" in item:
                shutil.rmtree(f"{modules_base}/{item}", ignore_errors=True)
                print(f"  Removed modules cache: {item}")

    # 3. Download + patch HUB cache
    from huggingface_hub import snapshot_download
    hub_dir = snapshot_download(
        MODEL_ID,
        allow_patterns=["*.py", "config.json", "tokenizer*", "special_tokens*", "chat_template*"],
        token=HF_TOKEN,
        force_download=True,
    )
    hub_py = f"{hub_dir}/modeling_nemotron_h.py"
    patch_nemotron_file(hub_py)

    # 4. Forçar transformers a copiar para modules cache + patch novamente
    # Trick: chamar get_class_from_dynamic_module vai copiar do hub para modules
    try:
        from transformers.dynamic_module_utils import get_class_from_dynamic_module
        # Vai disparar copy e depois patch aplica
        try:
            _ = get_class_from_dynamic_module(
                "modeling_nemotron_h.NemotronHForCausalLM",
                MODEL_ID,
                token=HF_TOKEN,
            )
        except Exception:
            pass  # pode falhar no import mas já copiou file
    except Exception as e:
        print(f"  warn get_class_from_dynamic_module: {e}")

    # 5. Patch modules cache também
    modules_py_paths = glob.glob(
        os.path.expanduser("~/.cache/huggingface/modules/transformers_modules/**/modeling_nemotron_h.py"),
        recursive=True,
    )
    print(f"Found {len(modules_py_paths)} modules cache files to patch:")
    for p in modules_py_paths:
        patch_nemotron_file(p)

    # 6. Clear sys.modules AGAIN para forçar re-exec do patched file
    for mod_name in list(sys.modules):
        if "nemotron" in mod_name.lower() or "Nemotron" in mod_name:
            del sys.modules[mod_name]

    print("[OK] Patch aplicado em ambos caches")

print("\nLoading Nemotron-3-Nano-30B-A3B em NF4 (3-5min)...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)

tok = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True,
)

if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

vram_used = torch.cuda.memory_allocated() / 1e9
print(f"[OK] Model NF4 loaded.")
print(f"  Model class: {type(model).__name__}")
print(f"  MAX_SEQ: {MAX_SEQ}")
print(f"  GPU mem used: {vram_used:.1f}GB")

Applying pure-PyTorch fallback patch in AMBOS caches (hub + modules)...


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

  [OK] patched (V1): /root/.cache/huggingface/hub/models--unsloth--Nemotron-3-Nano-30B-A3B/snapshots/2cea3c74d09568ab80edcffec2dc2383f661a610/modeling_nemotron_h.py
Found 1 modules cache files to patch:
  [OK] patched (V1): /root/.cache/huggingface/modules/transformers_modules/unsloth/Nemotron_hyphen_3_hyphen_Nano_hyphen_30B_hyphen_A3B/2cea3c74d09568ab80edcffec2dc2383f661a610/modeling_nemotron_h.py
[OK] Patch aplicado em ambos caches

Loading Nemotron-3-Nano-30B-A3B em NF4 (3-5min)...


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

[OK] Model NF4 loaded.
  Model class: NemotronHForCausalLM
  MAX_SEQ: 2048
  GPU mem used: 18.8GB


In [9]:
# Cell 5: LoRA PEFT direto - kienngx baseline (SEM MoE BOMBA por estabilidade)
from peft import LoraConfig, get_peft_model, TaskType

# 8 target_modules aceitos pelo Kaggle submission gate (kg1_submission_gate.py)
# - in_proj: Mamba-2 (OBRIGATORIO no gate)
# - out_proj: Mamba-2
# - q/k/v/o_proj: Attention layers
# - up/down_proj: MLP (shared expert, nao os 128 experts do MoE)
# NAO inclui: gate_proj, x_proj (rejeitados pelo gate - unconverted Mamba)
TARGET_MODULES_NEMOTRON = [
    "in_proj", "out_proj",
    "q_proj", "k_proj", "v_proj", "o_proj",
    "up_proj", "down_proj",
]

lora_config = LoraConfig(
    r=32,                            # kienngx
    lora_alpha=32,                   # kienngx (ratio 1:1)
    lora_dropout=0.05,               # kienngx
    bias="none",
    target_modules=TARGET_MODULES_NEMOTRON,
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
)

model = get_peft_model(model, lora_config)
model.enable_input_require_grads()  # necessario para grad_ckpt com PEFT

print("[OK] LoRA applied via PEFT direct (kienngx baseline)")
model.print_trainable_parameters()
# Esperado: ~80-100M trainable || ~30B all || trainable% ~0.30%

import torch
vram_used = torch.cuda.memory_allocated() / 1e9
print(f"GPU mem after LoRA: {vram_used:.1f}GB")


[OK] LoRA applied via PEFT direct (kienngx baseline)
trainable params: 883,873,792 || all params: 32,461,811,136 || trainable%: 2.7228
GPU mem after LoRA: 22.3GB


In [10]:
# Cell 6: Dataset train.csv OFICIAL Kaggle - kienngx replica (1200 samples seed=42)
import os, shutil
import pandas as pd
from datasets import Dataset

# Tenta Drive cache primeiro
TRAIN_CSV = "/content/drive/MyDrive/kg1_train.csv"

# Fallback: download via Kaggle API
if not os.path.exists(TRAIN_CSV):
    print("train.csv nao achado no Drive, baixando via Kaggle API...")
    if not os.path.exists("/root/.kaggle/kaggle.json"):
        raise RuntimeError("Configure KAGGLE_USERNAME/KEY nos Colab Secrets ou coloque kaggle.json no Drive")
    rc = os.system("kaggle competitions download -c nvidia-nemotron-model-reasoning-challenge -f train.csv -p /content/ 2>&1")
    if rc != 0:
        print("WARN: Kaggle download falhou. Verificar credenciais + aceitar rules da competition.")
    os.system("unzip -o /content/train.csv.zip -d /content/ 2>/dev/null || true")
    if os.path.exists("/content/train.csv"):
        TRAIN_CSV = "/content/train.csv"
        # Cache no Drive para proxima execucao
        shutil.copy(TRAIN_CSV, "/content/drive/MyDrive/kg1_train.csv")
        print("[OK] train.csv cached no Drive")

assert os.path.exists(TRAIN_CSV), f"train.csv nao encontrado em {TRAIN_CSV}"
print(f"Loading {TRAIN_CSV}")

df_full = pd.read_csv(TRAIN_CSV)
print(f"Full dataset: {len(df_full)} rows | columns: {list(df_full.columns)}")

# kienngx EXATO: sample(n=1200, seed=42)
SUBSAMPLE_SIZE = 1200
df = df_full.sample(n=SUBSAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f"Subsampled: {len(df)} rows (seed=42)")

# Auto-detect colunas
PROMPT_COL = "prompt" if "prompt" in df.columns else "problem"
ANSWER_COL = "answer" if "answer" in df.columns else "solution"
assert PROMPT_COL in df.columns, f"Coluna prompt nao encontrada em {df.columns}"
assert ANSWER_COL in df.columns, f"Coluna answer nao encontrada em {df.columns}"

# Prompt template kienngx EXATO
PROMPT_SUFFIX = chr(10) + "Put your final answer inside \\boxed{}."

def format_kienngx(row):
    user_msg = str(row[PROMPT_COL]) + PROMPT_SUFFIX
    assistant_msg = str(row[ANSWER_COL])  # raw answer (contem CoT + boxed)
    messages = [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_msg},
    ]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

ds_train = Dataset.from_pandas(df).map(
    format_kienngx,
    num_proc=2,
    remove_columns=list(df.columns),
)

print(f"[OK] Formatted: {len(ds_train)} examples")
print(f"Sample text (first 400 chars):")
print(ds_train[0]["text"][:400])


train.csv nao achado no Drive, baixando via Kaggle API...
[OK] train.csv cached no Drive
Loading /content/train.csv
Full dataset: 9500 rows | columns: ['id', 'prompt', 'answer']
Subsampled: 1200 rows (seed=42)


Map (num_proc=2):   0%|          | 0/1200 [00:00<?, ? examples/s]

[OK] Formatted: 1200 examples
Sample text (first 400 chars):
<|im_start|>system
<|im_end|>
<|im_start|>user
In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
01001000 -> 10100100
00011100 -> 11001110
01001001 -> 10110100
00011111 -> 11111111
01000100 


In [11]:
# Cell 7: SFT Training - kienngx EXATA (TRL 0.16+)
from trl import SFTTrainer, SFTConfig
import threading, time, trl
print(f"Using TRL {trl.__version__}")

args = SFTConfig(
    output_dir=CKPT_DIR,
    # === kienngx EXATO ===
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=1e-4,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=100,
    save_total_limit=3,
    bf16=True,
    optim="adamw_torch",
    max_length=MAX_SEQ,              # 2048 kienngx
    dataset_text_field="text",
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=1.0,
    report_to="none",
    push_to_hub=False,
    seed=42,
    remove_unused_columns=True,
    dataloader_num_workers=0,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds_train,
    args=args,
    processing_class=tok,
)

# Memory monitoring thread
import torch
def monitor_mem():
    while True:
        try:
            m = torch.cuda.memory_allocated() / 1e9
            p = torch.cuda.max_memory_allocated() / 1e9
            print(f"[MEM] current={m:.1f}GB peak={p:.1f}GB")
        except Exception:
            pass
        time.sleep(300)  # 5min
threading.Thread(target=monitor_mem, daemon=True).start()

# Resume from checkpoint se existir (Colab disconnect recovery)
resume = None
if os.path.exists(CKPT_DIR):
    ckpts = [d for d in os.listdir(CKPT_DIR) if d.startswith("checkpoint-")]
    if ckpts:
        resume = True
        print(f"Resuming from existing checkpoint (found {len(ckpts)} ckpts in {CKPT_DIR})")

print("Starting training (expected 6-8h in H100, 8-10h in A100)...")
stats = trainer.train(resume_from_checkpoint=resume)
print(f"Training done. Stats: {stats}")
print(f"Final loss: {stats.training_loss:.4f} (esperado: 0.5-1.5 kienngx)")


Using TRL 0.24.0


Adding EOS to train dataset:   0%|          | 0/1200 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1200 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1200 [00:00<?, ? examples/s]

[MEM] current=21.6GB peak=61.5GB
Starting training (expected 6-8h in H100, 8-10h in A100)...


Step,Training Loss
10,55.742731


[MEM] current=29.2GB peak=61.5GB


KeyboardInterrupt: 

In [ ]:
# Cell 8: Save adapter + validate submission gate + gerar submission.zip + upload HF
import os, json, zipfile
from huggingface_hub import HfApi

FINAL_DIR = f"{CKPT_DIR}/final_adapter"
trainer.save_model(FINAL_DIR)
tok.save_pretrained(FINAL_DIR)
print(f"[OK] Final adapter saved: {FINAL_DIR}")
print(f"Files: {os.listdir(FINAL_DIR)}")

# === Validacao gate Kaggle (alinhado com kg1_submission_gate.py) ===
print(chr(10) + "=== VALIDACAO submission gate ===")
adapter_config_path = f"{FINAL_DIR}/adapter_config.json"
if not os.path.exists(adapter_config_path):
    raise RuntimeError(f"adapter_config.json nao foi gerado em {FINAL_DIR}")

with open(adapter_config_path) as f:
    cfg = json.load(f)

target_modules = cfg.get("target_modules", [])
rank = cfg.get("r", cfg.get("lora_rank", 0))
print(f"target_modules: {target_modules}")
print(f"rank: {rank}")

errors = []
if not isinstance(target_modules, list) or not target_modules:
    errors.append("target_modules nao e list ou vazia")
else:
    if "in_proj" not in target_modules: errors.append("missing in_proj")
    if "gate_proj" in target_modules: errors.append("contains gate_proj (rejeitado gate)")
    if "x_proj" in target_modules: errors.append("contains x_proj (rejeitado gate)")
if rank > 32: errors.append(f"rank {rank} > 32 (vLLM limit)")

if errors:
    print(f"!!! GATE FAIL: {errors}")
else:
    print("[OK] adapter PASSA submission_gate local")

# === Gerar submission.zip (formato oficial Kaggle demo) ===
SUBMISSION_ZIP = f"{CKPT_DIR}/submission.zip"
REQUIRED = ["adapter_config.json", "adapter_model.safetensors"]

with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in REQUIRED:
        src = os.path.join(FINAL_DIR, f)
        if not os.path.exists(src):
            print(f"!!! WARN: {f} nao encontrado em {FINAL_DIR}")
            continue
        zf.write(src, arcname=f)
        size_mb = os.path.getsize(src) / 1e6
        print(f"  added: {f} ({size_mb:.1f} MB)")

size_mb = os.path.getsize(SUBMISSION_ZIP) / 1e6
print(f"[OK] submission.zip salvo: {SUBMISSION_ZIP} ({size_mb:.1f} MB)")

# === Upload HF (backup + compartilhamento) ===
api = HfApi(token=HF_TOKEN)
REPO_ID = "felipesp1983/kg1-nemotron-lora-v73-kienngx-final"
try:
    api.create_repo(REPO_ID, private=True, exist_ok=True)
    api.upload_folder(folder_path=FINAL_DIR, repo_id=REPO_ID, path_in_repo="final")
    api.upload_file(path_or_fileobj=SUBMISSION_ZIP, repo_id=REPO_ID, path_in_repo="submission.zip")
    print(f"[OK] Uploaded to HF: https://huggingface.co/{REPO_ID}")
except Exception as e:
    print(f"!!! WARN: HF upload falhou: {e}")
    print(f"Download manual: {SUBMISSION_ZIP} (via Colab Files pane)")

print(chr(10) + "=" * 60)
print("PROXIMOS PASSOS - KAGGLE SUBMIT:")
print("=" * 60)
print(f"1. Download submission.zip de: {SUBMISSION_ZIP}")
print("   (Colab Files pane -> download manual)")
print("2. No terminal local:")
print("   export KAGGLE_USERNAME=felipe1983 KAGGLE_KEY=<seu_key>")
print("   kaggle competitions submit \\")
print("       -c nvidia-nemotron-model-reasoning-challenge \\")
print("       -f submission.zip \\")
print(f'       -m "v73 kienngx final r=32 all-linear lr=1e-4 loss {stats.training_loss:.3f}"')
print("3. Aguardar score (~1-3h processamento Kaggle)")
print("4. Score esperado: 0.86 +/- 0.01")


## Troubleshooting (se algo falhar)

### Cell 1: `AssertionError: VRAM < 24GB`
- Runtime -> Change runtime type -> A100 High-RAM (Pro+) ou H100

### Cell 3: `AssertionError: Invalid HF_TOKEN`
- Clique no icone chave esquerda (Colab Secrets)
- Add secret: `HF_KEY` = seu token HF (comeca `hf_`)
- Toggle "Notebook access" ON

### Cell 4: Download timeout
- Normal demorar 3-5min (63GB download)
- Se falhar: Runtime -> Restart -> rodar Cell 1-4 novamente

### Cell 4: `GPU mem used > 30GB`
- NF4 nao aplicou. Provavel versao bitsandbytes incompativel.
- Fix: `!pip install -U bitsandbytes` + restart

### Cell 6: `train.csv nao encontrado`
- Configure `KAGGLE_USERNAME` + `KAGGLE_KEY` nos Secrets
- OU: coloque train.csv manualmente em /content/drive/MyDrive/kg1_train.csv

### Cell 7: OOM durante treino
- Impossivel com config atual (peak ~25GB). Se ocorrer, problema externo.
- Reduzir: `max_length` de 2048 -> 1024

### Cell 7: Colab disconnect durante treino
- Sem problema: Cell 7 auto-resume do ultimo checkpoint
- Basta re-rodar Cell 7 (depois de Cell 1-6 de novo)

### Cell 8: `GATE FAIL`
- Verifique adapter_config.json tem `in_proj` + rank <= 32
- Sem `gate_proj`/`x_proj`

## Validacao pre-submit (regra 99%)

Antes de submeter ao Kaggle, confirme:
1. `[OK] adapter PASSA submission_gate local` em Cell 8
2. Final loss em range 0.5-1.5 (muito acima/abaixo = problema)
3. submission.zip tem exatamente 2 arquivos (adapter_config.json + adapter_model.safetensors)
4. Tamanho submission.zip entre 200MB-500MB (tipico kienngx)

Se tudo OK: queime 1 submit slot Kaggle (limit 5/dia).
Score esperado: **0.86 +/- 0.01** (replica kienngx proven).

## Pipeline pos-v73

### Se atingir 0.86:
- **v74**: curriculum learning (easy -> hard) via huikang difficulty data -> 0.87
- **v75**: DARE-TIES merge com huikang_v26 adapter -> 0.87-0.88
- **v76**: GRPO stage-2 sobre hard subset -> 0.88+

### Se < 0.85:
- Investigar config antes de outro submit
- Verificar loss curve (deve descer de ~3.5 -> ~1.0)
- Verificar target_modules no adapter_config.json